In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/272.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/270.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/253.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/260.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/273.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/265.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/271.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/251.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/274.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/256.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/254.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/267.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/252.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/258.jpg
/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/test/0/261.jpg
/kaggle/input/datasets/ra

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# check dataset path
print(os.listdir("/kaggle/input"))

# define transforms
transform = transforms.Compose([
    transforms.Resize((128,128)), #makes all images of same size

    transforms.RandomHorizontalFlip(),   # flip image
    transforms.RandomRotation(10),       # rotate slightly
    
    transforms.ToTensor(),  #converts images into numbers
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)) #normalizes for better output
         
])

# load dataset
train_dataset = datasets.ImageFolder(
    root="/kaggle/input/datasets/razinw/dog-vs-cat/dogvscat/train",
    transform = transform
)

# create dataloader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# check classes
print(train_dataset.class_to_idx)

# check one batch
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

['datasets']
{'0': 0, '1': 1}
torch.Size([32, 3, 128, 128])
torch.Size([32])


In [3]:
import torch
import torch.nn as nn 

class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3,32,3)
        self.conv2 = nn.Conv2d(32,64,3)

        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.LazyLinear(128)
        self.fc2 = nn.Linear(128, 2)   # 2 classes

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        #print(x.shape)
        x = torch.flatten(x,1)
        x = self.dropout(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [4]:
model = CatDogCNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

In [8]:
for epoch in range(20):
    total_loss = 0
    for images, labels in train_loader:
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    avg_loss = total_loss/ len(train_loader)
    print(f"epochs : {epoch+1}, Loss: {avg_loss:.4f}")

epochs : 1, Loss: 0.1412
epochs : 2, Loss: 0.1681
epochs : 3, Loss: 0.1785
epochs : 4, Loss: 0.1596
epochs : 5, Loss: 0.1139
epochs : 6, Loss: 0.1047
epochs : 7, Loss: 0.1141
epochs : 8, Loss: 0.0831
epochs : 9, Loss: 0.0868
epochs : 10, Loss: 0.0871
epochs : 11, Loss: 0.1286
epochs : 12, Loss: 0.0728
epochs : 13, Loss: 0.0593
epochs : 14, Loss: 0.0724
epochs : 15, Loss: 0.0723
epochs : 16, Loss: 0.0986
epochs : 17, Loss: 0.0579
epochs : 18, Loss: 0.0454
epochs : 19, Loss: 0.0403
epochs : 20, Loss: 0.0515


In [6]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in train_loader:

        outputs = model(images)
        _, predicted = torch.max(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Accuracy:", correct/total)

Accuracy: 0.95
